In [ ]:
import numpy as np
import json
import cvxpy as cp
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict
import scipy.sparse as sp

from src.problems.utils import sample_data_for_group

In [ ]:
sns.set_style("white")
mpl.rc('font', **{'size': 18})
plt.rcParams["font.family"] = "Times New Roman"
sns.set_palette("tab10")

In [ ]:
N_CONSUMERS = 625
N_PRODUCERS = 625
GROUP_KEY = "top_category"
K_REC = 10
SOLVER = cp.GUROBI
GAMMA = 0.5
ALPHA = 0.95
DATA_PATH_ROOT = Path("../../data")

In [ ]:
def compute_consumer_optimal_solution_cvar(
    rel_matrix: np.ndarray,
    k_rec: int,
    producer_max_min_val: float,
    theta: float,
    group_assignments: list[int],
    alpha: float,
    prod_vals: np.ndarray,
    solver: str,
) -> tuple[float, np.ndarray]:
    """
    Given a relevance matrix, this function returns the convex optimization problem
    that minimizes the Conditional Value at Risk (CVaR) of the groups of consumer utility given a fixed number
    of producers to recommend.

    The problem is solved using CVaR (Conditional Value at Risk) approach.
    The CVaR is defined as the average of the worst-case losses, where the worst-case losses are defined
    as the losses that exceed a certain threshold (1 - alpha).
    """

    # producer allocations
    C, P = rel_matrix.shape

    # 1) consumer‑greedy baseline
    greedy = np.sort(rel_matrix, axis=1)[:, -k_rec:].sum(axis=1)  # (C,)

    # 2) sparse group‐indicator G (shape G×C), with 1/|G_i| weights
    unique_groups, inv = np.unique(group_assignments, return_inverse=True)
    G = len(unique_groups)
    sizes = np.bincount(inv)
    data = 1.0 / sizes[inv]  # length C
    G_sparse = sp.csr_matrix((data, (inv, np.arange(C))), shape=(G, C))

    rel_c = cp.Constant(rel_matrix)
    Gc = cp.Constant(G_sparse)
    g_c = cp.Constant(greedy)

    x = cp.Variable((C, P), boolean=True)
    rho = cp.Variable(nonneg=True)
    t = cp.Variable(G, nonneg=True)

    u = cp.sum(cp.multiply(rel_c, x), axis=1)
    loss_c = 1 - u / g_c
    loss_g = Gc @ loss_c

    constraints = [
        cp.sum(x, axis=1) == k_rec,
        cp.sum(cp.multiply(x.sum(axis=0), prod_vals)) >= (producer_max_min_val * theta),
        t >= loss_g - rho,
    ]
    cvar_obj = rho + (1.0 / ((1 - alpha) * G)) * cp.sum(t)

    prob = cp.Problem(cp.Minimize(cvar_obj), constraints)
    prob.solve(solver=cp.GUROBI, warm_start=True, **{"MIPGap": 1e-3})
    # prob.solve(solver=solver)

    return prob.value, x.value

In [ ]:
def compute_consumer_optimal_solution_mean(
    rel_matrix: np.ndarray, prod_vals: np.ndarray, k_rec: int, producer_max_min_val: float, theta: float, solver: str
) -> tuple[float, np.ndarray]:
    """
    Given a relevance matrix, this function returns the convex optimization problem
    that maximizes the mean consumer utility given a fixed number of producers to recommend.
    """

    allocations = cp.Variable(rel_matrix.shape, boolean=True)
    # constraints
    constraints = [
        # recommend k producers
        cp.sum(allocations, axis=1) == k_rec,
        # minimal producer utility must be at least gamma * producer_max_min_utility
        cp.sum(cp.multiply(allocations.sum(axis=0), prod_vals)) >= (producer_max_min_val * theta),
    ]

    # maximize the consumer objective
    problem = cp.Problem(
        cp.Maximize(cp.mean(cp.sum(cp.multiply(allocations, rel_matrix), axis=1))),
        constraints
    )
    problem.solve(solver=solver)

    return problem.value, problem.variables()[0].value

In [ ]:
allocations_for_theta_k = lambda theta, k_rec, prod_max_min_val: compute_consumer_optimal_solution_mean(
            rel_matrix=rel_matrix_sampled,
            prod_vals=prod_vals,
            k_rec=k_rec,
            producer_max_min_val=prod_max_min_val,
            theta=theta,
            solver=SOLVER
        )

allocations_for_theta_k_cvar = lambda theta, k_rec, prod_max_min_val: compute_consumer_optimal_solution_cvar(
            rel_matrix=rel_matrix_sampled,
            prod_vals=prod_vals,
            k_rec=k_rec,
            producer_max_min_val=prod_max_min_val,
            theta=theta,
            group_assignments=group_assignments,
            alpha=ALPHA,
            solver=SOLVER
            )


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict
from matplotlib.lines import Line2D

# Load data
with open(DATA_PATH_ROOT / "movielens_predictions.npy", "rb") as f:
    REL_MATRIX = np.load(f)

with open(DATA_PATH_ROOT / "movielens_user_groups.json", "r") as f:
    GROUPS_MAP = json.load(f)

# Sample a subset of consumers/producers for plotting
rel_matrix_sampled, consumer_ids, group_assignments = sample_data_for_group(
    n_consumers=625,
    n_producers=625,
    groups_map=GROUPS_MAP,
    group_key=GROUP_KEY,
    data=REL_MATRIX,
    naive_sampling=True,
    seed=1,
)

prod_vals = rel_matrix_sampled.sum(axis=0).max() - rel_matrix_sampled.sum(axis=0)
prod_max_min_val = sum(sorted(prod_vals)[::-1][:K_REC]) * 625

_, mean_allocations_0 = allocations_for_theta_k(
    theta=0,
    k_rec=K_REC,
    prod_max_min_val=prod_max_min_val,
)
_, mean_allocations_05 = allocations_for_theta_k(
    theta=0.5,
    k_rec=K_REC,
    prod_max_min_val=prod_max_min_val,
)
_, mean_allocations_1 = allocations_for_theta_k(
    theta=1,
    k_rec=K_REC,
    prod_max_min_val=prod_max_min_val,
)

# assume grid1, grid2, grid3 are your 10×10 numpy arrays
# and you want vmin=0, vmax=100, cmap='crest_r', zeros in black as before

grid1 = mean_allocations_0.sum(axis=0).reshape((25, 25))
grid2 = mean_allocations_05.sum(axis=0).reshape((25, 25))
grid3 = mean_allocations_1.sum(axis=0).reshape((25, 25))

cmap = sns.color_palette("flare_r", as_cmap=True)
# make 0 to be black
cmap.set_under('black')

# 1) make a figure with 4 columns: 3 heatmaps + 1 colorbar
fig = plt.figure(figsize=(12, 4), dpi=300)
gs  = GridSpec(1, 4, width_ratios=[1, 1, 1, 0.05], wspace=0.3)

# 2) create your three axes for the heatmaps
axes = [fig.add_subplot(gs[0, i]) for i in range(3)]

# 3) plot the first two with no colorbar
for ax, grid in zip(axes[:2], (grid1, grid2)):
    sns.heatmap(
        grid,
        ax=ax,
        cbar=False,
        cmap=cmap,        # your crest_r with set_under('black') from earlier
        vmin=0.1, vmax=100,
        square=True,      # enforces each cell is square
        xticklabels=0,
        yticklabels=0,
        linewidths=0.5,
        linecolor='black',
    )
    ax.title.set_text("$\\theta = $" + str(0 if ax == axes[0] else 0.5))

# 4) create the colorbar axis
cax = fig.add_subplot(gs[0, 3])

# 5) plot the third heatmap + colorbar, pointing at cax
sns.heatmap(
    grid3,
    ax=axes[2],
    cbar_ax=cax,
    cbar_kws={"label": "Producer utility share %"},
    cmap=cmap,
    vmin=0.1, vmax=100,
    square=True,
    xticklabels=0,
    yticklabels=0,
    linewidths=0.5,
    linecolor='black',
    )
axes[2].title.set_text(f"$\\theta = {1.0}$")
# add a bit of space below titles


for ax in axes + [cax]:
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1)  # adjust thickness if desired


# 6) export for LaTeX
plt.savefig("three_heatmaps_theta.pdf", bbox_inches="tight")


In [ ]:
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict
from matplotlib.lines import Line2D

# Load data
with open(DATA_PATH_ROOT / "movielens_predictions.npy", "rb") as f:
    REL_MATRIX = np.load(f)

with open(DATA_PATH_ROOT / "movielens_user_groups.json", "r") as f:
    GROUPS_MAP = json.load(f)

# Sample a subset of consumers/producers for plotting
rel_matrix_sampled, consumer_ids, group_assignments = sample_data_for_group(
    n_consumers=625,
    n_producers=625,
    groups_map=GROUPS_MAP,
    group_key=GROUP_KEY,
    data=REL_MATRIX,
    naive_sampling=True,
    seed=1,
)

prod_vals = rel_matrix_sampled.sum(axis=0).max() - rel_matrix_sampled.sum(axis=0)
max_gmv = sum(sorted(prod_vals)[::-1][:625])  # maximum GMV for the top 625 producers


# Prepare two result dictionaries: one for CVaR-based allocations, one for mean-based allocations
res_cvar = defaultdict(dict)
res_mean = defaultdict(dict)

for k_rec in [10, 20, 100]:
    print(k_rec)
    for theta in [0, 0.1, 0.35, 0.5, 0.75, 0.95]:
        prod_max_min_val = sum(sorted(prod_vals)[::-1][:k_rec]) * 625
        # Compute allocation matrices for CVaR and for mean
        _, cvar_alls = allocations_for_gamma_k_cvar(theta, k_rec, prod_max_min_val)
        _, mean_alls = allocations_for_gamma_k(theta, k_rec, prod_max_min_val)

        # For CVaR allocations
        top_picks_cvar = []
        chosens = []
        # Make a copy of the cvar allocation matrix so we can zero out allocations after a pick
        cvar_allocs_matrix = cvar_alls.copy()
        for consumer_id in range(rel_matrix_sampled.shape[0]):
            consumer_allocs = cvar_allocs_matrix[consumer_id, :] * rel_matrix_sampled[consumer_id, :]
            top_allocs_idx = np.argsort(consumer_allocs)[-k_rec:][::-1]
            # Sample picks
            draws = np.random.binomial(n=1, p=consumer_allocs[top_allocs_idx])
            picks = draws * top_allocs_idx
            picks = picks[picks != 0]
            if picks.size > 0:
                chosen = picks[0]
                top_picks_cvar.append(consumer_allocs[chosen])
                chosens.append(chosen)
                cvar_allocs_matrix[:, chosen] = 0  # remove that producer from further picks

        mean_consumer_utility_cvar = np.mean(top_picks_cvar)
        gmv = sum(prod_vals[chosens])

        res_cvar[k_rec][theta] = {
            "mean_c_util": mean_consumer_utility_cvar,
            "gmv": gmv / max_gmv,  # normalize GMV by maximum GMV
        }

        # For mean allocations
        top_picks_mean = []
        chosens = []
        # Copy mean allocation matrix for zeroing out
        mean_allocs_matrix = mean_alls.copy()
        for consumer_id in range(rel_matrix_sampled.shape[0]):
            consumer_allocs = mean_allocs_matrix[consumer_id, :] * rel_matrix_sampled[consumer_id, :]
            top_allocs_idx = np.argsort(consumer_allocs)[-k_rec:][::-1]
            # Sample picks
            draws = np.random.binomial(n=1, p=consumer_allocs[top_allocs_idx])
            picks = draws * top_allocs_idx
            picks = picks[picks != 0]
            if picks.size > 0:
                chosen = picks[0]
                top_picks_mean.append(consumer_allocs[chosen])
                chosens.append(chosen)
                mean_allocs_matrix[:, chosen] = 0  # remove that producer

        mean_consumer_utility_mean = np.mean(top_picks_mean)
        # take the value of top_picks_mean indexes from prod_vals
        gmv = sum(prod_vals[chosens])

        res_mean[k_rec][theta] = {
            "mean_c_util": mean_consumer_utility_mean,
            "gmv": gmv / max_gmv,  # normalize GMV by maximum GMV
        }

# Plotting: one subplot per k_rec, with four curves each:
#  - mean consumer utility (mean allocations)
#  - mean consumer utility (CVaR allocations)
#  - STR (mean allocations)
#  - STR (CVaR allocations)



k_recs = list(res_cvar.keys())
N = len(k_recs)

fig, axes = plt.subplots(1, N, figsize=(4 * N + 2, 4), dpi=300, sharex=False)
if N == 1:
    axes = [axes]

for i, k_rec in enumerate(k_recs):
    thetas = sorted(res_cvar[k_rec].keys())

    # Extract y-values for each curve
    # CVaR
    mean_utils_cvar = [res_cvar[k_rec][theta]["mean_c_util"] for theta in thetas]
    gmv_cvar = [res_cvar[k_rec][theta]["gmv"] for theta in thetas]
    # Mean
    mean_utils_mean = [res_mean[k_rec][theta]["mean_c_util"] for theta in thetas]
    gmv_mean = [res_mean[k_rec][theta]["gmv"] for theta in thetas]

    ax1 = axes[i]

    ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    ax1.grid(which='both', axis='both', linestyle='--', alpha=0.4)

    ax1.plot(
        thetas,
        gmv_mean,
        marker="s",
        linestyle="-",
        linewidth=2,
        color="#4A567E",
        label="Mean"
    )
    ax1.plot(
        thetas,
        gmv_cvar,
        marker="s",
        linestyle="-",
        linewidth=2,
        color="#e36a5d",
        label="CVaR"
    )
    #if i == 0:
        #ax1.legend(markerscale=0)
    #ax1.set_ylabel("STR")
    ax1.set_title(f"k = {k_rec}")
    y_min, y_max = ax1.get_ylim()
    x_min, x_max = ax1.get_xlim()
    if y_min < 0:
        y_min = 0
    if y_max > 1:
        y_max = 1
    ax1.set_yticks(np.linspace(y_min, y_max, 3))
    ax1.set_xticks(np.linspace(0, 1, 3))

fig.supxlabel(
        r"Fraction of max sum of producer value guaranteed, $\theta$",
        x=0.5, y=-0, ha="center"
    )
fig.supylabel(
        "GMV",
        x=0.01, y=0.5, va="center"
    )


plt.tight_layout()
plt.savefig("consumer_vs_theta_mean_and_cvar_movielens.pdf", bbox_inches="tight")

In [ ]:
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict
from matplotlib.lines import Line2D

# Load data
with open(DATA_PATH_ROOT / "simrec1_predictions.npy", "rb") as f:
    REL_MATRIX = np.load(f)

with open(DATA_PATH_ROOT / "simrec1_user_groups.json", "r") as f:
    GROUPS_MAP = json.load(f)

# Sample a subset of consumers/producers for plotting
rel_matrix_sampled, consumer_ids, group_assignments = sample_data_for_group(
    n_consumers=625,
    n_producers=625,
    groups_map=GROUPS_MAP,
    group_key=GROUP_KEY,
    data=REL_MATRIX,
    naive_sampling=True,
    seed=1,
)

prod_vals = rel_matrix_sampled.sum(axis=0).max() - rel_matrix_sampled.sum(axis=0)
max_gmv = sum(sorted(prod_vals)[::-1][:625])  # maximum GMV for the top 625 producers


# Prepare two result dictionaries: one for CVaR-based allocations, one for mean-based allocations
res_cvar = defaultdict(dict)
res_mean = defaultdict(dict)

for k_rec in [10, 20, 100]:
    print(k_rec)
    for theta in [0, 0.1, 0.35, 0.5, 0.75, 0.95]:
        prod_max_min_val = sum(sorted(prod_vals)[::-1][:k_rec]) * 625
        # Compute allocation matrices for CVaR and for mean
        _, cvar_alls = allocations_for_gamma_k_cvar(theta, k_rec, prod_max_min_val)
        _, mean_alls = allocations_for_gamma_k(theta, k_rec, prod_max_min_val)

        # For CVaR allocations
        top_picks_cvar = []
        chosens = []
        # Make a copy of the cvar allocation matrix so we can zero out allocations after a pick
        cvar_allocs_matrix = cvar_alls.copy()
        for consumer_id in range(rel_matrix_sampled.shape[0]):
            consumer_allocs = cvar_allocs_matrix[consumer_id, :] * rel_matrix_sampled[consumer_id, :]
            top_allocs_idx = np.argsort(consumer_allocs)[-k_rec:][::-1]
            # Sample picks
            draws = np.random.binomial(n=1, p=consumer_allocs[top_allocs_idx])
            picks = draws * top_allocs_idx
            picks = picks[picks != 0]
            if picks.size > 0:
                chosen = picks[0]
                top_picks_cvar.append(consumer_allocs[chosen])
                chosens.append(chosen)
                cvar_allocs_matrix[:, chosen] = 0  # remove that producer from further picks

        mean_consumer_utility_cvar = np.mean(top_picks_cvar)
        gmv = sum(prod_vals[chosens])

        res_cvar[k_rec][theta] = {
            "mean_c_util": mean_consumer_utility_cvar,
            "gmv": gmv / max_gmv,  # normalize GMV by maximum GMV
        }

        # For mean allocations
        top_picks_mean = []
        chosens = []
        # Copy mean allocation matrix for zeroing out
        mean_allocs_matrix = mean_alls.copy()
        for consumer_id in range(rel_matrix_sampled.shape[0]):
            consumer_allocs = mean_allocs_matrix[consumer_id, :] * rel_matrix_sampled[consumer_id, :]
            top_allocs_idx = np.argsort(consumer_allocs)[-k_rec:][::-1]
            # Sample picks
            draws = np.random.binomial(n=1, p=consumer_allocs[top_allocs_idx])
            picks = draws * top_allocs_idx
            picks = picks[picks != 0]
            if picks.size > 0:
                chosen = picks[0]
                top_picks_mean.append(consumer_allocs[chosen])
                chosens.append(chosen)
                mean_allocs_matrix[:, chosen] = 0  # remove that producer

        mean_consumer_utility_mean = np.mean(top_picks_mean)
        # take the value of top_picks_mean indexes from prod_vals
        gmv = sum(prod_vals[chosens])

        res_mean[k_rec][theta] = {
            "mean_c_util": mean_consumer_utility_mean,
            "gmv": gmv / max_gmv,  # normalize GMV by maximum GMV
        }

# Plotting: one subplot per k_rec, with four curves each:
#  - mean consumer utility (mean allocations)
#  - mean consumer utility (CVaR allocations)
#  - STR (mean allocations)
#  - STR (CVaR allocations)



k_recs = list(res_cvar.keys())
N = len(k_recs)

fig, axes = plt.subplots(1, N, figsize=(4 * N + 2, 4), dpi=300, sharex=False)
if N == 1:
    axes = [axes]

for i, k_rec in enumerate(k_recs):
    thetas = sorted(res_cvar[k_rec].keys())

    # Extract y-values for each curve
    # CVaR
    mean_utils_cvar = [res_cvar[k_rec][theta]["mean_c_util"] for theta in thetas]
    gmv_cvar = [res_cvar[k_rec][theta]["gmv"] for theta in thetas]
    # Mean
    mean_utils_mean = [res_mean[k_rec][theta]["mean_c_util"] for theta in thetas]
    gmv_mean = [res_mean[k_rec][theta]["gmv"] for theta in thetas]

    ax1 = axes[i]

    ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    ax1.grid(which='both', axis='both', linestyle='--', alpha=0.4)

    ax1.plot(
        thetas,
        gmv_mean,
        marker="s",
        linestyle="-",
        linewidth=2,
        color="#4A567E",
        label="Mean"
    )
    ax1.plot(
        thetas,
        gmv_cvar,
        marker="s",
        linestyle="-",
        linewidth=2,
        color="#e36a5d",
        label="CVaR"
    )
    #if i == 0:
        #ax1.legend(markerscale=0)
    #ax1.set_ylabel("STR")
    ax1.set_title(f"k = {k_rec}")
    y_min, y_max = ax1.get_ylim()
    x_min, x_max = ax1.get_xlim()
    if y_min < 0:
        y_min = 0
    if y_max > 1:
        y_max = 1
    ax1.set_yticks(np.linspace(y_min, y_max, 3))
    ax1.set_xticks(np.linspace(0, 1, 3))

fig.supxlabel(
        r"Fraction of max sum of producer value guaranteed, $\theta$",
        x=0.5, y=-0, ha="center"
    )
fig.supylabel(
        "GMV",
        x=0.01, y=0.5, va="center"
    )


plt.tight_layout()
plt.savefig("consumer_vs_theta_mean_and_cvar_simrec.pdf", bbox_inches="tight")

In [ ]:
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict
from matplotlib.lines import Line2D

# Load data
with open(DATA_PATH_ROOT / "amazon_predictions.npy", "rb") as f:
    REL_MATRIX = np.load(f)

with open(DATA_PATH_ROOT / "amazon_user_groups.json", "r") as f:
    GROUPS_MAP = json.load(f)

# Sample a subset of consumers/producers for plotting
rel_matrix_sampled, consumer_ids, group_assignments = sample_data_for_group(
    n_consumers=625,
    n_producers=625,
    groups_map=GROUPS_MAP,
    group_key=GROUP_KEY,
    data=REL_MATRIX,
    naive_sampling=True,
    seed=1,
)

prod_vals = rel_matrix_sampled.sum(axis=0).max() - rel_matrix_sampled.sum(axis=0)
max_gmv = sum(sorted(prod_vals)[::-1][:625])  # maximum GMV for the top 625 producers


# Prepare two result dictionaries: one for CVaR-based allocations, one for mean-based allocations
res_cvar = defaultdict(dict)
res_mean = defaultdict(dict)

for k_rec in [10, 20, 100]:
    print(k_rec)
    for theta in [0, 0.1, 0.35, 0.5, 0.75, 0.95]:
        prod_max_min_val = sum(sorted(prod_vals)[::-1][:k_rec]) * 625
        # Compute allocation matrices for CVaR and for mean
        _, cvar_alls = allocations_for_gamma_k_cvar(theta, k_rec, prod_max_min_val)
        _, mean_alls = allocations_for_gamma_k(theta, k_rec, prod_max_min_val)

        # For CVaR allocations
        top_picks_cvar = []
        chosens = []
        # Make a copy of the cvar allocation matrix so we can zero out allocations after a pick
        cvar_allocs_matrix = cvar_alls.copy()
        for consumer_id in range(rel_matrix_sampled.shape[0]):
            consumer_allocs = cvar_allocs_matrix[consumer_id, :] * rel_matrix_sampled[consumer_id, :]
            top_allocs_idx = np.argsort(consumer_allocs)[-k_rec:][::-1]
            # Sample picks
            draws = np.random.binomial(n=1, p=consumer_allocs[top_allocs_idx])
            picks = draws * top_allocs_idx
            picks = picks[picks != 0]
            if picks.size > 0:
                chosen = picks[0]
                top_picks_cvar.append(consumer_allocs[chosen])
                chosens.append(chosen)
                cvar_allocs_matrix[:, chosen] = 0  # remove that producer from further picks

        mean_consumer_utility_cvar = np.mean(top_picks_cvar)
        gmv = sum(prod_vals[chosens])

        res_cvar[k_rec][theta] = {
            "mean_c_util": mean_consumer_utility_cvar,
            "gmv": gmv / max_gmv,  # normalize GMV by maximum GMV
        }

        # For mean allocations
        top_picks_mean = []
        chosens = []
        # Copy mean allocation matrix for zeroing out
        mean_allocs_matrix = mean_alls.copy()
        for consumer_id in range(rel_matrix_sampled.shape[0]):
            consumer_allocs = mean_allocs_matrix[consumer_id, :] * rel_matrix_sampled[consumer_id, :]
            top_allocs_idx = np.argsort(consumer_allocs)[-k_rec:][::-1]
            # Sample picks
            draws = np.random.binomial(n=1, p=consumer_allocs[top_allocs_idx])
            picks = draws * top_allocs_idx
            picks = picks[picks != 0]
            if picks.size > 0:
                chosen = picks[0]
                top_picks_mean.append(consumer_allocs[chosen])
                chosens.append(chosen)
                mean_allocs_matrix[:, chosen] = 0  # remove that producer

        mean_consumer_utility_mean = np.mean(top_picks_mean)
        # take the value of top_picks_mean indexes from prod_vals
        gmv = sum(prod_vals[chosens])

        res_mean[k_rec][theta] = {
            "mean_c_util": mean_consumer_utility_mean,
            "gmv": gmv / max_gmv,  # normalize GMV by maximum GMV
        }

# Plotting: one subplot per k_rec, with four curves each:
#  - mean consumer utility (mean allocations)
#  - mean consumer utility (CVaR allocations)
#  - STR (mean allocations)
#  - STR (CVaR allocations)



k_recs = list(res_cvar.keys())
N = len(k_recs)

fig, axes = plt.subplots(1, N, figsize=(4 * N + 2, 4), dpi=300, sharex=False)
if N == 1:
    axes = [axes]

for i, k_rec in enumerate(k_recs):
    thetas = sorted(res_cvar[k_rec].keys())

    # Extract y-values for each curve
    # CVaR
    mean_utils_cvar = [res_cvar[k_rec][theta]["mean_c_util"] for theta in thetas]
    gmv_cvar = [res_cvar[k_rec][theta]["gmv"] for theta in thetas]
    # Mean
    mean_utils_mean = [res_mean[k_rec][theta]["mean_c_util"] for theta in thetas]
    gmv_mean = [res_mean[k_rec][theta]["gmv"] for theta in thetas]

    ax1 = axes[i]

    ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    ax1.grid(which='both', axis='both', linestyle='--', alpha=0.4)

    ax1.plot(
        thetas,
        gmv_mean,
        marker="s",
        linestyle="-",
        linewidth=2,
        color="#4A567E",
        label="Mean"
    )
    ax1.plot(
        thetas,
        gmv_cvar,
        marker="s",
        linestyle="-",
        linewidth=2,
        color="#e36a5d",
        label="CVaR"
    )
    if i == 0:
        ax1.legend(markerscale=0)
    #ax1.set_ylabel("STR")
    ax1.set_title(f"k = {k_rec}")
    y_min, y_max = ax1.get_ylim()
    x_min, x_max = ax1.get_xlim()
    if y_min < 0:
        y_min = 0
    if y_max > 1:
        y_max = 1
    ax1.set_yticks(np.linspace(y_min, y_max, 3))
    ax1.set_xticks(np.linspace(0, 1, 3))

fig.supxlabel(
        r"Fraction of max sum of producer value guaranteed, $\theta$",
        x=0.5, y=-0, ha="center"
    )
fig.supylabel(
        "GMV",
        x=0.01, y=0.5, va="center"
    )


plt.tight_layout()
plt.savefig("consumer_vs_theta_mean_and_cvar_amazon.pdf", bbox_inches="tight")

In [ ]:
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict
from matplotlib import ticker as mticker

# Load data
with open(DATA_PATH_ROOT / "movielens_predictions.npy", "rb") as f:
    REL_MATRIX = np.load(f)

with open(DATA_PATH_ROOT / "movielens_user_groups.json", "r") as f:
    GROUPS_MAP = json.load(f)

GROUP_KEY = "top_category"
K_REC = 10
ALPHA = 0.95
SOLVER = cp.GUROBI

# Sample a subset of consumers/producers for plotting
rel_matrix_sampled, consumer_ids, group_assignments = sample_data_for_group(
    n_consumers=625,
    n_producers=625,
    groups_map=GROUPS_MAP,
    group_key=GROUP_KEY,
    data=REL_MATRIX,
    naive_sampling=True,
    seed=1,
)

# Prepare two result dictionaries: one for CVaR-based allocations, one for mean-based allocations
res_cvar = defaultdict(dict)
res_mean = defaultdict(dict)

# sort rel_matrix_sampled for each row, take top K_REC elements
top_allocs = np.sort(rel_matrix_sampled, axis=1)[:, -K_REC:][:, ::-1].sum(axis=1)
prod_vals = rel_matrix_sampled.sum(axis=0).max() - rel_matrix_sampled.sum(axis=0)
prod_max_min_val = sum(sorted(prod_vals)[::-1][:K_REC]) * 625

res = {}
for theta in [0, 0.1, 0.35, 0.5, 0.75, 1]:
    _, cvar_alls = allocations_for_theta_k_cvar(theta, K_REC, prod_max_min_val)
    _, mean_alls = allocations_for_theta_k(theta, K_REC, prod_max_min_val)
    cvar_util = np.min((cvar_alls * rel_matrix_sampled).sum(axis=1) / top_allocs)
    mean_util = np.min((mean_alls * rel_matrix_sampled).sum(axis=1) / top_allocs)
    res[theta] = {
        "mean_c_util_cvar": cvar_util,
        "mean_c_util_mean": mean_util,
    }


fig = plt.figure(figsize=(5, 4), dpi=300)
mean_res = []
cvar_res = []
thetas = sorted(res.keys())
for theta in res:
    mean_res.append([res[theta]["mean_c_util_mean"]])
    cvar_res.append([res[theta]["mean_c_util_cvar"]])

plt.grid(True, linestyle="--", alpha=0.4)
plt.plot(
    thetas,
    cvar_res,
    marker="s",
    label="CVaR",
    color="#e36a5d",
    linewidth=2,
)
plt.plot(
    thetas,
    mean_res,
    marker="s",
    label="Mean",
    color="#4A567E",
    linewidth=2,
)
plt.xticks([0, 0.5, 1])
y_min, y_max = plt.ylim()
plt.yticks(np.linspace(y_min, 1, 3))
plt.gca().yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
#plt.title("Allocations Comparison")
plt.xlabel("$\\theta$")
plt.ylabel("Consumer utility")
plt.legend(markerscale=0, loc="lower left")
plt.tight_layout()
plt.savefig("mean_diffs_theta_movielens.pdf", bbox_inches="tight")